<a href="https://colab.research.google.com/github/AR-Ashik-9997/Phitron-practice-problem/blob/main/LSTM_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [256]:
! pip install nltk
!pip install torchmetrics

In [257]:
import nltk
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter
from torch.utils.data import Dataset,DataLoader
from nltk.tokenize import word_tokenize
from torchmetrics.classification import Accuracy

In [258]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [259]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [260]:
document = """The Last Train Journey

On a cold winter evening, Daniel arrived at the old railway station carrying a small leather bag.
The station was nearly empty except for a few passengers waiting quietly on wooden benches.
A loud whistle echoed through the air as the final train of the night slowly entered the platform.

Daniel checked the ticket in his pocket and walked toward the train.
The metal doors opened with a creaking sound and warm air rushed outside.
Inside the train, yellow lights flickered softly above rows of empty seats.

He chose a seat beside the window and placed his bag carefully near his feet.
As the train began to move, the city lights slowly disappeared behind thick clouds of fog.
Snow started falling gently outside while distant mountains appeared under the moonlight.

Across the aisle sat an elderly woman reading an old book with a blue cover.
Every few minutes she looked outside the window as if searching for something familiar.
Daniel noticed a silver necklace around her neck shaped like a small compass.

After several hours, the train stopped at a remote station surrounded by tall pine trees.
Very few people entered or left the train at that place.
A strange silence covered the station as the doors closed once again.

Curious about the woman, Daniel finally started a conversation.
The woman introduced herself as Eleanor and explained that she was returning to her hometown after many years.
She said the town was hidden deep within the northern mountains and was known for its beautiful frozen lake.

As the journey continued, Eleanor shared stories from her childhood.
She described colorful festivals, music performances, and lanterns floating across the lake during winter nights.
Daniel listened carefully while the train moved through snowy valleys and dark forests.

Near midnight, the train suddenly slowed down because heavy snow had covered the tracks ahead.
Passengers became nervous and began discussing possible delays.
Conductors moved through the cabins trying to calm everyone.

Daniel looked outside and saw workers removing snow under bright floodlights.
The freezing wind shook the train slightly while snow continued falling from the dark sky.
Despite the delay, Eleanor remained calm and continued telling her stories.

Several hours later, the tracks were finally cleared and the train resumed its journey.
The passengers sighed with relief as warm coffee was served throughout the cabins.
Daniel and Eleanor continued talking about travel, history, and forgotten places around the world.

Just before sunrise, the train reached the mountain town.
The station was small but beautifully decorated with glowing lamps and snow-covered rooftops.
People wearing thick winter coats walked slowly across the icy streets.

Before leaving, Eleanor handed Daniel a folded piece of paper.
Inside it was a drawing of the frozen lake along with a short message thanking him for the conversation.
Daniel smiled and carefully placed the paper inside his notebook.

As the train departed once again, Daniel stood on the platform watching the sunrise over the mountains.
The cold air, quiet streets, and distant church bells created a peaceful atmosphere he would never forget.
Years later, Daniel would still remember that mysterious winter journey and the stories shared by Eleanor on the last train of the night.
"""

In [261]:
# Tokenize
tokens=word_tokenize(document.lower())

In [262]:
# Build Vocab
vocab={'<unk>':0}
for token in Counter(tokens).keys():
  if token not in vocab:
    vocab[token]=len(vocab)

In [263]:
# text to numeric
def text_to_numeric(sentence,vocab):
  return [vocab.get(token,vocab['<unk>']) for token in sentence]


In [264]:
input_sentence=document.split('\n')
input_numerical_sentence = []
for sentence in input_sentence:
  input_numerical_sentence.append(text_to_numeric(word_tokenize(sentence.lower()),vocab))

In [265]:
#training Sequence form
training_sequence=[]
for sentence in input_numerical_sentence:
  for i in range(1,len(sentence)):
    training_sequence.append(sentence[:i+1])

In [266]:
#padding
max_len=max(len(seq) for seq in training_sequence)
padded_training_sequence=torch.tensor([[0]* (max_len-len(seq)) + seq for seq in training_sequence],dtype=torch.long)

In [267]:
# spliting padded Tensor
X=padded_training_sequence[:,:-1]
y=padded_training_sequence[:,-1]

In [268]:
# Custom Dataset
class CustomDataset(Dataset):
  def __init__(self,X,y):
    self.X=X
    self.y=y

  def __len__(self):
    return self.X.shape[0]

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

In [269]:
# Dataset and Dataloder
dataset=CustomDataset(X,y)
dataloader=DataLoader(dataset,batch_size=32,shuffle=True)

In [270]:
# LSTM Model Definition
class LSTMMdel(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.embedding=nn.Embedding(vocab_size,100)
    self.lstm=nn.LSTM(100,150,batch_first=True)
    self.fc=nn.Linear(150,vocab_size)

  def forward(self,x):
    embedded=self.embedding(x)
    _,(final_hidden_state,_)=self.lstm(embedded)
    output=self.fc(final_hidden_state.squeeze(0))
    return output

In [271]:
model=LSTMMdel(len(vocab)).to(device)

## Training Loop

In [272]:
epochs=50
learning_rate=0.001
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=learning_rate)

In [273]:
for epoch in range(epochs):
    total_loss = 0.0
    for batch_x, batch_y in dataloader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    # 'epoc' er bodole 'epoch' kora hoyeche
    print(f"Epoch: {epoch+1} | Loss: {total_loss:.4f}")

Epoch: 1 | Loss: 101.0349
Epoch: 2 | Loss: 93.9200
Epoch: 3 | Loss: 87.7533
Epoch: 4 | Loss: 83.0393
Epoch: 5 | Loss: 76.9332
Epoch: 6 | Loss: 70.9293
Epoch: 7 | Loss: 66.1898
Epoch: 8 | Loss: 61.3744
Epoch: 9 | Loss: 55.8843
Epoch: 10 | Loss: 50.2465
Epoch: 11 | Loss: 45.2696
Epoch: 12 | Loss: 40.8861
Epoch: 13 | Loss: 36.6574
Epoch: 14 | Loss: 32.1333
Epoch: 15 | Loss: 28.9238
Epoch: 16 | Loss: 25.5543
Epoch: 17 | Loss: 22.5081
Epoch: 18 | Loss: 19.9899
Epoch: 19 | Loss: 17.6983
Epoch: 20 | Loss: 15.6060
Epoch: 21 | Loss: 13.9088
Epoch: 22 | Loss: 12.6524
Epoch: 23 | Loss: 11.2530
Epoch: 24 | Loss: 10.1586
Epoch: 25 | Loss: 9.3139
Epoch: 26 | Loss: 8.4376
Epoch: 27 | Loss: 7.8707
Epoch: 28 | Loss: 7.2065
Epoch: 29 | Loss: 6.7184
Epoch: 30 | Loss: 6.1283
Epoch: 31 | Loss: 5.9458
Epoch: 32 | Loss: 5.5393
Epoch: 33 | Loss: 5.0043
Epoch: 34 | Loss: 4.6928
Epoch: 35 | Loss: 4.4596
Epoch: 36 | Loss: 4.2954
Epoch: 37 | Loss: 4.1274
Epoch: 38 | Loss: 4.0336
Epoch: 39 | Loss: 3.6997
Epoch: 40

Prediction

In [274]:
def prediction(model,vocab,text):
  model.eval()
  tokenize_text=word_tokenize(text.lower())
  numerical_text=text_to_numeric(tokenize_text,vocab)
  padding_text=torch.tensor([0]*(25-len(numerical_text))+numerical_text,dtype=torch.long).unsqueeze(0).to(device)
  output=model(padding_text)
  value,index=torch.max(output,dim=1)
  return text + " "+list(vocab.keys())[index]

In [275]:
prediction(model,vocab,"Curious about the woman , daniel")

'Curious about the woman , daniel finally'

In [276]:
def calculate_accuracy(model,dataloader,device,vocab_size):
  model.eval()
  metrics=Accuracy(task="multiclass",num_classes=vocab_size).to(device)
  with torch.no_grad():
    for batch_x,batch_y in dataloader:
      batch_x,batch_y=batch_x.to(device),batch_y.to(device)
      output=model(batch_x)
      metrics.update(output,batch_y)
  total_accuracy=metrics.compute().item()*100
  return total_accuracy

In [277]:
accuracy=calculate_accuracy(model,dataloader,device,len(vocab))

In [278]:
print(f"Model Accuracy {accuracy :.2f}%")

Model Accuracy 96.94%


# Generate Automatically

In [279]:
import time
num_tokens=20
input_text='The Last Train Journey On a cold winter evening'
for i in range(num_tokens):
  output_text=prediction(model,vocab,input_text)
  input_text=output_text
  time.sleep(0.5)
  print(output_text)

The Last Train Journey On a cold winter evening ,
The Last Train Journey On a cold winter evening , daniel
The Last Train Journey On a cold winter evening , daniel arrived
The Last Train Journey On a cold winter evening , daniel arrived at
The Last Train Journey On a cold winter evening , daniel arrived at the
The Last Train Journey On a cold winter evening , daniel arrived at the old
The Last Train Journey On a cold winter evening , daniel arrived at the old railway
The Last Train Journey On a cold winter evening , daniel arrived at the old railway station
The Last Train Journey On a cold winter evening , daniel arrived at the old railway station carrying
The Last Train Journey On a cold winter evening , daniel arrived at the old railway station carrying a
The Last Train Journey On a cold winter evening , daniel arrived at the old railway station carrying a small
The Last Train Journey On a cold winter evening , daniel arrived at the old railway station carrying a small leather
The La